In [1]:
import yaml

with open("C:/Users/Lales/.dbt/profiles.yml", 'r') as stream:
    data_loaded = yaml.safe_load(stream)

DBCONFIG = data_loaded["classicmodels_modeling"]["outputs"]["dev"]

DBHOST = DBCONFIG["host"]
DBPORT = int(DBCONFIG["port"])
DBNAME = DBCONFIG["dbname"]
DBUSER = DBCONFIG["user"]
DBPASSWORD = DBCONFIG["pass"]

db_connection_url = f'postgresql+psycopg2://{DBUSER}:{DBPASSWORD}@{DBHOST}:{DBPORT}/{DBNAME}'

In [2]:
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(db_connection_url)

df = pd.read_sql("""
SELECT *
FROM information_schema.tables
WHERE table_schema = 'public'
""", engine)

df

,table_catalog,table_schema,table_name,table_type,self_referencing_column_name,reference_generation,user_defined_type_catalog,user_defined_type_schema,user_defined_type_name,is_insertable_into,is_typed,commit_action
0,Student,public,productlines,BASE TABLE,None,None,None,None,None,YES,NO,None
1,Student,public,products,BASE TABLE,None,None,None,None,None,YES,NO,None
2,Student,public,employees,BASE TABLE,None,None,None,None,None,YES,NO,None
3,Student,public,offices,BASE TABLE,None,None,None,None,None,YES,NO,None
4,Student,public,customers,BASE TABLE,None,None,None,None,None,YES,NO,None
5,Student,public,payments,BASE TABLE,None,None,None,None,None,YES,NO,None
6,Student,public,orders,BASE TABLE,None,None,None,None,None,YES,NO,None
7,Student,public,orderdetails,BASE TABLE,None,None,None,None,None,YES,NO,None


### 1-Read your PostgreSQL tables

In [3]:
customers = pd.read_sql("""SELECT * FROM public.customers""", engine)
employees = pd.read_sql("""SELECT * FROM public.employees""", engine)
offices = pd.read_sql("""SELECT * FROM public.offices""", engine)
payments = pd.read_sql("""SELECT * FROM public.payments""", engine)
products = pd.read_sql("""SELECT * FROM public.products""", engine)
product_lines = pd.read_sql("""SELECT * FROM public.productlines""", engine)
orders = pd.read_sql("""SELECT * FROM public.orders""", engine)
order_details = pd.read_sql("""SELECT * FROM public.orderdetails""", engine)


### 2-Save as Parquet

In [5]:
customers.to_parquet("customers.parquet")
products.to_parquet("products.parquet")
orders.to_parquet("orders.parquet")
order_details.to_parquet("order_details.parquet")
employees.to_parquet("employees.parquet")
offices.to_parquet("offices.parquet")
payments.to_parquet("payments.parquet")
product_lines.to_parquet("product_lines.parquet")

### 3-Upload to Amazon S3

aws s3 cp customers.parquet s3://pyspark-demo-lala/raw/customers/

aws s3 cp products.parquet s3://pyspark-demo-lala/raw/products/

aws s3 cp orders.parquet s3://pyspark-demo-lala/raw/orders/

aws s3 cp order_details.parquet s3://pyspark-demo-lala/raw/order_details/

aws s3 cp employees.parquet s3://pyspark-demo-lala/raw/employees/

aws s3 cp offices.parquet s3://pyspark-demo-lala/raw/offices/

aws s3 cp payments.parquet s3://pyspark-demo-lala/raw/payments/

aws s3 cp product_lines.parquet s3://pyspark-demo-lala/raw/product_lines/




### 4-Process with PySpark in AWS Glue

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

customers = spark.read.parquet("s3://your-bucket/raw/customers/")
orders = spark.read.parquet("s3://your-bucket/raw/orders/")
order_items = spark.read.parquet("s3://your-bucket/raw/order_items/")
products = spark.read.parquet("s3://your-bucket/raw/products/")

sales = (
    orders.join(customers, "customer_id")
          .join(order_items, "order_id")
          .join(products, "product_id")
)

sales.show()

In [ ]:
from pyspark.sql.functions import monotonically_increasing_id

dim_customer = (
    customers
    .withColumn("customer_key", monotonically_increasing_id())
    .select(
        "customer_key",
        "customer_id",
        "first_name",
        "last_name",
        "city",
        "state"
    )
)

In [ ]:
fact_sales = (
    orders
    .join(order_items, "order_id")
    .join(dim_customer, "customer_id")
    .join(products, "product_id")
    .select(
        "customer_key",
        "product_id",
        "order_date",
        "quantity",
        "unit_price"
    )
)

In [ ]:
dim_customer.write.mode("overwrite").parquet(
    "s3://your-bucket/warehouse/dim_customer"
)

fact_sales.write.mode("overwrite").parquet(
    "s3://your-bucket/warehouse/fact_sales"
)